# Stage 4: Model Explainability (SHAP)
# Here we load the trained model, run an inference test, and generate a SHAP waterfall chart to dissect the mathematical logic behind the prediction.

In [ ]:
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# 1. Load Test Data and the Trained Model
df = pd.read_csv('data/processed_signals.csv')
features = ['yield_curve_slope', 'credit_gdp', 'credit_gdp_diff2', 'credit_gdp_cycle', 'yield_curve_cycle', 'cpi', 'unemp', 'debtgdp']

model = xgb.XGBClassifier()
model.load_model("models/xgboost_crisis_model.json")

# 2. Isolate a specific test case (e.g., USA right before the 2008 crash)
test_case = df[(df['country'] == 'USA') & (df['year'] == 2007)][features]

# 3. Run Inference
prob = model.predict_proba(test_case)[0][1]
print(f"Predicted Crisis Risk: {prob * 100:.2f}%")

# 4. Generate SHAP Diagnostics
explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
shap_values = explainer(test_case)

# Plot the waterfall chart
plt.style.use('default')
fig, ax = plt.subplots(figsize=(8, 5))
shap.plots.waterfall(shap_values[0])